[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/run_all.ipynb)

# ▶ Run All — full PdM + Anomaly Detection pipeline

This single notebook runs the **entire project end-to-end** in one click:
data → features → predictive maintenance → anomaly detection. It bootstraps the
environment **once**, then flows straight through every stage.

> Use this when you want to *run everything*. For the detailed, step-by-step
> teaching version of each stage, open the numbered notebooks `00`–`03`.

**In Colab:** `Runtime → Run all` (Ctrl/Cmd+F9). On the free GPU the whole thing
finishes in a few minutes.

## 0 · Bootstrap (clone repo + install deps) — runs once

This clones the repo and installs dependencies on Colab, and puts the repo root on
the import path. Everything below reuses the code in `src/`.

In [ ]:
# --- Environment bootstrap (works locally AND on Google Colab) ---------------
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab there is no repo yet, so clone it and install dependencies.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Make the repo root importable so `from src import ...` works from notebooks/.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("running on Colab" if IN_COLAB else "running locally")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

from src import data, features, models, utils
np.random.seed(0)
results = {}   # collect headline metrics for the final summary
print("imports OK")

## 1 · Load both datasets

- **AI4I 2020** (UCI) — tabular, labelled → supervised failure classification.
- **NASA C-MAPSS FD001** — run-to-failure turbofan time series → RUL + anomaly.

Both download and cache automatically.

In [ ]:
ai4i = data.load_ai4i()
cm = data.load_cmapss("FD001")
print("AI4I:", ai4i.shape, "| C-MAPSS train:", cm["train"].shape,
      "test:", cm["test"].shape)
print("AI4I failure rate: %.2f%%" % (100 * ai4i["Machine failure"].mean()))

## 2 · Feature engineering (C-MAPSS)

Add the clipped **RUL** label, drop dead (constant) sensors, add **rolling**
mean/std trend features, and build **sequence windows** for the LSTM. (Full
explanation of *why* each step in notebook `01`.)

In [ ]:
cols = features.feature_columns(cm["train"])
train_fe = features.add_rolling_features(
    features.add_rul(cm["train"], clip=125), cols)
X_seq, y_seq = features.make_sequences(train_fe, cols, seq_len=30)
scaler = utils.Standardizer().fit(X_seq.reshape(-1, X_seq.shape[-1]))
scale = lambda a: scaler.transform(a.reshape(-1, a.shape[-1])).reshape(a.shape)
X_seq_s = scale(X_seq)
print("active features:", len(cols), "| sequence tensor:", X_seq_s.shape)

## 3 · Predictive Maintenance — Part A: failure classification (AI4I)

A class-balanced **Random Forest** predicts the failure flag. We report the
confusion matrix, precision/recall (what matters under 96/4 imbalance), and
ROC-AUC, plus feature importance. (Detail in notebook `02`.)

In [ ]:
X, y = features.prepare_ai4i(ai4i)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                       random_state=42, stratify=y)
clf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                             random_state=42, n_jobs=-1).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]
pred = clf.predict(Xte)
auc = roc_auc_score(yte, proba)
results["AI4I classification ROC-AUC"] = round(auc, 3)

print("Confusion matrix [true x pred]:\n", confusion_matrix(yte, pred))
print("\n", classification_report(yte, pred, digits=3))
print("ROC-AUC: %.3f" % auc)

imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(figsize=(7, 4), title="AI4I feature importance")
plt.tight_layout(); plt.show()

## 4 · Predictive Maintenance — Part B: RUL regression (C-MAPSS LSTM)

A 2-layer **LSTM** learns degradation trends and predicts Remaining Useful Life.
We score on held-out test engines with **RMSE** and the asymmetric **C-MAPSS
score** (punishes optimistic predictions harder). Detail in notebook `02`.

In [ ]:
model = models.LSTMRegressor(n_features=X_seq_s.shape[-1], hidden=64, layers=2)
hist = models.train_lstm(model, X_seq_s, y_seq, epochs=15, batch_size=256)
utils.plot_loss(hist, "LSTM training (MSE)"); plt.show()

X_test_seq = features.last_sequence_per_unit(cm["test"], cols, seq_len=30)
y_pred = models.predict_lstm(model, scale(X_test_seq))
y_true = cm["rul"]["rul"].to_numpy().clip(max=125)
rmse = utils.rmse(y_true, y_pred)
results["C-MAPSS RUL RMSE (cycles)"] = round(rmse, 1)
print("Test RMSE: %.2f cycles | C-MAPSS score: %.1f"
      % (rmse, utils.cmapss_score(y_true, y_pred)))
utils.plot_rul_scatter(y_true, y_pred); plt.show()

## 5 · Anomaly Detection (unsupervised, C-MAPSS)

Train **Isolation Forest** and an **Autoencoder** on *healthy* data only, then check
they flag *degraded* readings — graded by ROC-AUC against the RUL we held back.
Detail in notebook `03`.

In [ ]:
df = train_fe
healthy = df[df.rul >= 100]; degraded = df[df.rul <= 20]
sc = utils.Standardizer().fit(healthy[cols].to_numpy("float32"))
Xh = sc.transform(healthy[cols].to_numpy("float32"))
Xd = sc.transform(degraded[cols].to_numpy("float32"))

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42).fit(Xh)
sh, sd = -iso.decision_function(Xh), -iso.decision_function(Xd)

ae = models.AutoEncoder(n_features=Xh.shape[1], latent=8)
models.train_autoencoder(ae, Xh, epochs=25, verbose=False)
eh, ed = models.reconstruction_error(ae, Xh), models.reconstruction_error(ae, Xd)

ylab = np.r_[np.zeros(len(Xh)), np.ones(len(Xd))]
results["Anomaly IsoForest AUC"] = round(roc_auc_score(ylab, np.r_[sh, sd]), 3)
results["Anomaly Autoencoder AUC"] = round(roc_auc_score(ylab, np.r_[eh, ed]), 3)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(sh, bins=40, alpha=.6, label="healthy", density=True)
ax[0].hist(sd, bins=40, alpha=.6, label="degraded", density=True)
ax[0].set_title("Isolation Forest score"); ax[0].legend()
ax[1].hist(eh, bins=40, alpha=.6, label="healthy", density=True)
ax[1].hist(ed, bins=40, alpha=.6, label="degraded", density=True)
ax[1].set_title("Autoencoder recon error"); ax[1].legend()
plt.show()

## 6 · Summary — everything in one table

Headline metrics for the full run. These are reproducible on a laptop CPU; numbers
vary slightly run-to-run due to model randomness.

In [ ]:
print("=== PIPELINE RESULTS ===")
for k, v in results.items():
    print(f"  {k:32s}: {v}")

### Done ✅

You just ran the complete ML core: predictive maintenance (classification + RUL) and
unsupervised anomaly detection on open IIoT datasets.

**Next:** the GenAI layer lives in the sibling project **`iiot-ai-rag`** — it consumes
these outputs (the autoencoder's bottleneck as a vector signature, plus failure/RUL
predictions) to retrieve similar past incidents from a vector DB and generate
maintenance work-orders with an LLM.